In [1]:
import multiprocessing as mp
import numpy as np
import time
from numba import jit

@jit(nopython=True)
def monte_carlo_chunk(num_samples_chunk):
    count_inside = 0
    for i in range(num_samples_chunk):
        x = np.random.random()
        y = np.random.random()
        if x**2 + y**2 <= 1.0:
            count_inside += 1
    return count_inside

def calculate_pi(num_samples, num_processes='Mult'):
    if num_processes == 'Mult':
        num_processes = mp.cpu_count()
    
        chunk_size = num_samples // num_processes
        chunks = [chunk_size] * num_processes
        chunks[-1] += num_samples % num_processes
        
        with mp.Pool(processes=num_processes) as pool:
            results = pool.map(monte_carlo_chunk, chunks)
        
        total_inside = sum(results)
    else:
        results = monte_carlo_chunk(num_samples)
    return 4.0 * total_inside / num_samples

num_samples = 100_000_000

print("=== Параллельное вычисление ===")
start_time = time.time()
pi_estimate = calculate_pi(num_samples)
end_time = time.time()

print(f"Расчитанное значение π: {pi_estimate:.8f}")
print(f"Время вычисления: {end_time - start_time:.4f}s")
print(f"Реальное значение π: {np.pi:.8f}")
print(f"Ошибка: {abs(pi_estimate - np.pi):.8f}")

print('------------------------------------------------')

print("=== Без параллельного вычисления ===")
start_time2 = time.time()
pi_estimate2 = calculate_pi(num_samples)
end_time2 = time.time()

print(f"Расчитанное значение π: {pi_estimate2:.8f}")
print(f"Время вычисления: {end_time2 - start_time2:.4f}s")
print(f"Реальное значение π: {np.pi:.8f}")
print(f"Ошибка: {abs(pi_estimate2 - np.pi):.8f}")

=== Параллельное вычисление ===
Расчитанное значение π: 3.14157008
Время вычисления: 2.0226s
Реальное значение π: 3.14159265
Ошибка: 0.00002257
------------------------------------------------
=== Без параллельного вычисления ===
Расчитанное значение π: 3.14162112
Время вычисления: 1.6214s
Реальное значение π: 3.14159265
Ошибка: 0.00002847


In [ ]:
import datetime
import json
import csv
import random
import string
from typing import List, Dict, Any
from pydantic import BaseModel, Field
import yaml

def generate_random_password() -> Dict[str, Any]:

    services = ["Google", "Facebook", "Twitter", "GitHub", "Microsoft", 
                "Apple", "Amazon", "Netflix", "Instagram", "LinkedIn",
                "Yandex", "Mail.ru", "VK", "Telegram", "WhatsApp",
                "Slack", "Discord", "Zoom", "Trello", "Notion"]
    
    descriptions = ["Рабочий аккаунт", "Личный аккаунт", "Для разработки", 
                   "Резервный аккаунт", "Тестовый аккаунт", "Корпоративный",
                   "Семейный доступ", "Для учебы", "Клиентский", "Админский"]
    
    hints = ["День рождения мамы", "Имя первой собаки", "Улица детства",
             "Любимый фильм", "Девичья фамилия мамы", "Первый учитель",
             "Номер первой машины", "Любимая книга", "Кличка кота"]
    
    name = f"{random.choice(services)}_{random.randint(1, 1000)}"
    
    description = random.choice(descriptions) if random.random() > 0.2 else None
    hint = random.choice(hints) if random.random() > 0.3 else None
    
    password_length = random.randint(8, 20)
    password = ''.join(random.choices(
        string.ascii_letters + string.digits + "!@#$%^&*", 
        k=password_length
    ))
    
    end_date = datetime.datetime.now()
    start_date = end_date - datetime.timedelta(days=365*2)
    
    random_days_created = random.randint(0, 365*2)
    created_at = start_date + datetime.timedelta(days=random_days_created)
    
    days_between = (end_date - created_at).days
    random_days_updated = random.randint(0, days_between)
    updated_at = created_at + datetime.timedelta(days=random_days_updated)
    
    return {
        "name": name,
        "description": description,
        "hint": hint,
        "password": password,
        "created_at": created_at,
        "updated_at": updated_at
    }


passwords_list = [generate_random_password() for _ in range(5000)]
print(f"Сгенерировано {len(passwords_list)} объектов")

class Password(BaseModel):
    name: str
    description: str | None = Field(default=None)
    hint: str | None = Field(default=None)
    password: str
    created_at: datetime.datetime
    updated_at: datetime.datetime
    
    class Config:
        json_encoders = {
            datetime.datetime: lambda v: v.isoformat()
        }

class Passwords(BaseModel):
    passwords: List[Password] = Field(default_factory=list)

print("\nСоздание объектов Pydantic...")
passwords_pydantic = Passwords(passwords=[
    Password(
        name=p["name"],
        description=p["description"],
        hint=p["hint"],
        password=p["password"],
        created_at=p["created_at"],
        updated_at=p["updated_at"]
    ) for p in passwords_list
])
print(f"Создано {len(passwords_pydantic.passwords)} объектов Pydantic")

print("\n" + "="*50)
print("РАБОТА С JSON (стандартный модуль)")
print("="*50)

def datetime_to_str(obj):
    if isinstance(obj, datetime.datetime):
        return obj.isoformat()
    return obj

passwords_for_json = []
for p in passwords_list:
    p_copy = p.copy()
    p_copy["created_at"] = p_copy["created_at"].isoformat()
    p_copy["updated_at"] = p_copy["updated_at"].isoformat()
    passwords_for_json.append(p_copy)

print("Запись в JSON файл...")
with open("passwords_standard.json", "w", encoding="utf-8") as f:
    json.dump({"passwords": passwords_for_json}, f, indent=2, 
              default=datetime_to_str, ensure_ascii=False)
print("Файл passwords_standard.json создан")

print("Чтение JSON файла...")
with open("passwords_standard.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    print(f"Загружено объектов: {len(data['passwords'])}")
    
    for p in data["passwords"][:2]:
        print(f"  Объект: {p['name']}, пароль: {p['password'][:10]}...")
        print(f"    created_at: {p['created_at']}")
        print(f"    updated_at: {p['updated_at']}")
        print()

print("\n" + "="*50)
print("РАБОТА С JSON (Pydantic)")
print("="*50)

print("Запись JSON через Pydantic...")
with open("passwords_pydantic.json", "w", encoding="utf-8") as f:
    json_data = passwords_pydantic.model_dump_json(indent=2)
    f.write(json_data)
print("Файл passwords_pydantic.json создан")

print("Чтение JSON через Pydantic...")
with open("passwords_pydantic.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    loaded_passwords = Passwords(**data)
    print(f"Загружено через Passwords(**data): {len(loaded_passwords.passwords)} объектов")

    f.seek(0)
    json_str = f.read()
    loaded_passwords2 = Passwords.model_validate_json(json_str)
    print(f"Загружено через model_validate_json: {len(loaded_passwords2.passwords)} объектов")
    
    if loaded_passwords.passwords:
        first = loaded_passwords.passwords[0]
        print(f"Первый объект: {first.name}")
        print(f"  Описание: {first.description}")
        print(f"  Пароль: {first.password[:10]}...")
        print(f"  Создан: {first.created_at}")

print("\n" + "="*50)
print("РАБОТА С YAML (стандартный модуль)")
print("="*50)

print("Запись в YAML файл...")
with open("passwords_standard.yaml", "w", encoding="utf-8") as f:
    passwords_for_yaml = []
    for p in passwords_list[:100]:
        p_copy = p.copy()
        p_copy["created_at"] = p_copy["created_at"].isoformat()
        p_copy["updated_at"] = p_copy["updated_at"].isoformat()
        passwords_for_yaml.append(p_copy)
    
    yaml.dump({"passwords": passwords_for_yaml}, f, default_flow_style=False, 
              allow_unicode=True)
print("Файл passwords_standard.yaml создан")

print("Чтение YAML файла...")
with open("passwords_standard.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)
    print(f"Загружено объектов: {len(data['passwords'])}")
    if data["passwords"]:
        print(f"Первый объект: {data['passwords'][0]['name']}")

print("\n" + "="*50)
print("РАБОТА С YAML (Pydantic)")
print("="*50)

print("Запись YAML через Pydantic...")
passwords_dicts = [p.model_dump() for p in passwords_pydantic.passwords[:100]]

with open("passwords_pydantic.yaml", "w", encoding="utf-8") as f:
    yaml.dump({"passwords": passwords_dicts}, f, default_flow_style=False, 
              allow_unicode=True)
print("Файл passwords_pydantic.yaml создан ")

print("Чтение YAML и создание объектов Pydantic...")
with open("passwords_pydantic.yaml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)
    for p in data["passwords"]:
        p["created_at"] = datetime.datetime.fromisoformat(str(p["created_at"]))
        p["updated_at"] = datetime.datetime.fromisoformat(str(p["updated_at"]))
    
    loaded_yaml_passwords = Passwords(passwords=data["passwords"])
    print(f"Создано объектов Pydantic из YAML: {len(loaded_yaml_passwords.passwords)}")

print("\n" + "="*50)
print("РАБОТА С CSV (стандартный модуль)")
print("="*50)

print("Запись в CSV файл...")
with open("passwords_standard.csv", "w", newline="", encoding="utf-8") as f:
    if passwords_list[:100]:
        passwords_for_csv = []
        for p in passwords_list[:200]:
            p_copy = p.copy()
            p_copy["created_at"] = p_copy["created_at"].isoformat()
            p_copy["updated_at"] = p_copy["updated_at"].isoformat()
            passwords_for_csv.append(p_copy)
        
        fieldnames = set()
        for p in passwords_for_csv:
            fieldnames.update(p.keys())
        fieldnames = list(fieldnames)
        
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter=";")
        writer.writeheader()
        writer.writerows(passwords_for_csv)
print("Файл passwords_standard.csv создан ")

print("Чтение CSV файла...")
with open("passwords_standard.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    rows = list(reader)
    print(f"Загружено строк: {len(rows)}")
    
    for i, row in enumerate(rows[:2]):
        print(f"Строка {i+1}:")
        for key, value in row.items():
            if value:
                print(f"  {key}: {value[:30]}{'...' if len(str(value)) > 30 else ''}")
        print()

print("\n" + "="*50)
print("РАБОТА С CSV (Pydantic)")
print("="*50)

print("Запись CSV через Pydantic...")
with open("passwords_pydantic.csv", "w", newline="", encoding="utf-8") as f:
    passwords_to_save = passwords_pydantic.passwords[:100]
    
    if passwords_to_save:
        passwords_dicts = [p.model_dump() for p in passwords_to_save]
        
        fieldnames = set()
        for p in passwords_dicts:
            fieldnames.update(p.keys())
        fieldnames = list(fieldnames)
        
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter=";")
        writer.writeheader()
        
        for p_dict in passwords_dicts:
            row = p_dict.copy()
            row["created_at"] = row["created_at"].isoformat()
            row["updated_at"] = row["updated_at"].isoformat()
            writer.writerow(row)
print("Файл passwords_pydantic.csv создан (первые 200 объектов)")

print("Чтение CSV и создание объектов Pydantic...")
with open("passwords_pydantic.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    rows = list(reader)
    
    csv_passwords = []
    for row in rows[:10]:
        processed_row = {}
        for key, value in row.items():
            if value == "":
                processed_row[key] = None
            elif key in ["created_at", "updated_at"]:
                processed_row[key] = datetime.datetime.fromisoformat(value)
            else:
                processed_row[key] = value
        
        try:
            password_obj = Password(**processed_row)
            csv_passwords.append(password_obj)
        except Exception as e:
            print(f"Ошибка при создании объекта: {e}")
            print(f"  Строка: {processed_row}")
    
    print(f"Успешно создано объектов Pydantic из CSV: {len(csv_passwords)}")
    if csv_passwords:
        print(f"Первый объект: {csv_passwords[0].name}")



Сгенерировано 5000 объектов

Создание объектов Pydantic...
Создано 5000 объектов Pydantic

РАБОТА С JSON (стандартный модуль)
Запись в JSON файл...
Файл passwords_standard.json создан
Чтение JSON файла...
Загружено объектов: 5000
  Объект: Facebook_912, пароль: TWge2nCF...
    created_at: 2024-12-20T00:05:55.701532
    updated_at: 2025-07-19T00:05:55.701532

  Объект: Discord_907, пароль: 0p%VR#Hoa...
    created_at: 2025-03-24T00:05:55.701552
    updated_at: 2025-08-04T00:05:55.701552


РАБОТА С JSON (Pydantic)
Запись JSON через Pydantic...
Файл passwords_pydantic.json создан
Чтение JSON через Pydantic...
Загружено через Passwords(**data): 5000 объектов
Загружено через model_validate_json: 5000 объектов
Первый объект: Facebook_912
  Описание: Личный аккаунт
  Пароль: TWge2nCF...
  Создан: 2024-12-20 00:05:55.701532

РАБОТА С YAML (стандартный модуль)
Запись в YAML файл...
Файл passwords_standard.yaml создан
Чтение YAML файла...
Загружено объектов: 100
Первый объект: Facebook_912

РАБО

C:\Users\arsen\AppData\Local\Temp\ipykernel_4864\2937862777.py:73: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Password(BaseModel):


Файл passwords_pydantic.yaml создан 
Чтение YAML и создание объектов Pydantic...
Создано объектов Pydantic из YAML: 100

РАБОТА С CSV (стандартный модуль)
Запись в CSV файл...
Файл passwords_standard.csv создан 
Чтение CSV файла...
Загружено строк: 200
Строка 1:
  updated_at: 2025-07-19T00:05:55.701532
  name: Facebook_912
  created_at: 2024-12-20T00:05:55.701532
  description: Личный аккаунт
  hint: Номер первой машины
  password: TWge2nCF

Строка 2:
  updated_at: 2025-08-04T00:05:55.701552
  name: Discord_907
  created_at: 2025-03-24T00:05:55.701552
  description: Семейный доступ
  hint: Любимый фильм
  password: 0p%VR#Hoa


РАБОТА С CSV (Pydantic)
Запись CSV через Pydantic...
Файл passwords_pydantic.csv создан (первые 200 объектов)
Чтение CSV и создание объектов Pydantic...
Успешно создано объектов Pydantic из CSV: 10
Первый объект: Facebook_912
